##DWH-3500

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set IsSecuritizedFlag =1
where portfolioid in (14,15,16,27,29,47,48,50,64,65)

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =66
where portfolioid=74

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =65
where portfolioid=73

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =64
where portfolioid=72

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =62
where portfolioid=70

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =61
where portfolioid=69

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set portfolio_id_optiverse =63
where portfolioid=71

In [0]:
%sql
update `p-cdh`.primaryactive.portfolio
set IsSecuritizedFlag =0
where portfolioid in (61,62)


## lookup_export table update

In [0]:
env="p"

In [0]:
%sql
drop table `{env}-cdh`.reference.lookup_export

In [0]:
%sql
create or replace table `{env}-cdh`.reference.lookup_export
as select * from `qa-cdh`.reference.lookup_export

## DExposure update

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, Unrealizedinterest, DueUnrealizedinterest
    FROM `{env}-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalUnrealizedInterest = coalesce(source.Unrealizedinterest, 0) + coalesce(source.DueUnrealizedinterest, 0);
""")
 

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, DuePrincipal, Dueinterest, DueUnrealizedinterest, DueExpense
    FROM `{env}-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.NetDueAmount = coalesce(source.DuePrincipal, 0) + coalesce(source.Dueinterest, 0)
                               + coalesce(source.DueUnrealizedinterest, 0) + coalesce(source.DueExpense, 0);
""")
 

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, interest, Dueinterest, DueUnrealizedinterest, Penaltyinterest
        , PenaltyUnrealizedinterest, Unrealizedinterest, Accrued, DueAccrued
    FROM `{env}-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalInterestAccruals = 
coalesce(source.interest, 0) + coalesce(source.Dueinterest, 0) +
coalesce(source.DueUnrealizedinterest, 0) + coalesce(source.Penaltyinterest, 0) 
+ coalesce(source.PenaltyUnrealizedinterest, 0) + coalesce(source.Unrealizedinterest, 0) +
coalesce(source.Accrued, 0) + coalesce(source.DueAccrued, 0);
""")

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, TotalInterest
    FROM `{env}-cdh`.derivativeactive.dexposure
    WHERE ExposureStatus = 'ΕΠΙΣΦΑΛΕΙΑ'
    AND SourceSystem = 'profits'
    AND (ProductKey like '%4015' or
         ProductKey like '%4016' or
         ProductKey like '%6078' or
         ProductKey like '%6079' or
         ProductKey like '%7362' or
         ProductKey like '%7363' or
         ProductKey like '%40000' or
         ProductKey like '%400001')
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalInterestL3869 = source.TotalInterest
""")

## DBTC-1295 
### Steps
* Rerun legal_process_step_qc for all businessdates in prod enviroment.